<a href="https://colab.research.google.com/github/nirajan-sah/ML/blob/Pytorch/Multiclass_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

In [2]:
from sklearn.datasets import make_circles

# Make 1000 samples
samples=1000

# Divide samples
X, y = make_circles(samples,
                    noise=0.03,
                    random_state=42)
DATA = X, y

In [11]:
import torch
from sklearn.model_selection import train_test_split


class Trainer:
    """
    Generic PyTorch trainer for multiclass classification.

    The model must output raw logits of shape (N, num_classes).
    Do not add a Softmax layer at the end of the model.

    Parameters
    ----------
    model : torch.nn.Module
        PyTorch model to train.

    loss_fn : callable
        Loss function, e.g. torch.nn.CrossEntropyLoss().

    DATA : tuple
        Tuple (X, y).
        X: features, shape (N, features).
        y: integer class labels, shape (N,). Not one-hot.

    random_state : int, default=42
        Seed for the train-test split and for training.

    display_range : int, default=10
        Metrics are printed every `display_range` epochs
        (and on the last epoch).

    epochs : int, default=100
        Number of training epochs.

    lr : float, default=0.001
        Learning rate for the Adam optimizer.

    train_size : float, default=0.8
        Fraction of data used for training.

    stratify : bool, default=True
        Keep the same class balance in the train and test sets.
    """

    def __init__(
        self,
        model,
        loss_fn,
        DATA,
        random_state=42,
        display_range=10,
        epochs=100,
        lr=0.001,
        train_size=0.8,
        stratify=True,
    ):
        self.device = self.get_device()
        self.model = model.to(self.device)

        self.loss_fn = loss_fn
        self.DATA = DATA

        self.random_state = random_state
        self.display_range = display_range
        self.epochs = epochs
        self.train_size = train_size
        self.stratify = stratify

        self.optimizer = torch.optim.Adam(
            self.model.parameters(),
            lr=lr,
        )

        self.history = {
            "epoch": [],
            "train_loss": [],
            "test_loss": [],
            "train_acc": [],
            "test_acc": [],
        }

        self.split_data()

    def get_device(self):
        """Return CUDA device if available, otherwise CPU."""
        return torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )

    def split_data(self):
        """
        Split the data into train and test sets and move them
        to the selected device.

        Features become float32 tensors.
        Labels become long tensors of shape (N,).
        """
        X, y = self.DATA

        (
            x_train,
            x_test,
            y_train,
            y_test,
        ) = train_test_split(
            X,
            y,
            train_size=self.train_size,
            random_state=self.random_state,
            shuffle=True,
            stratify=y if self.stratify else None,
        )

        self.x_train = torch.as_tensor(
            x_train, dtype=torch.float32
        ).to(self.device)

        self.x_test = torch.as_tensor(
            x_test, dtype=torch.float32
        ).to(self.device)

        # CrossEntropyLoss needs class indices (long), shape (N,)
        self.y_train = torch.as_tensor(
            y_train, dtype=torch.long
        ).view(-1).to(self.device)

        self.y_test = torch.as_tensor(
            y_test, dtype=torch.long
        ).view(-1).to(self.device)

    def set_epochs(self, epochs):
        """Update the number of training epochs."""
        self.epochs = epochs

    def set_lr(self, lr):
        """Update the optimizer learning rate."""
        for param_group in self.optimizer.param_groups:
            param_group["lr"] = lr

    @staticmethod
    def accuracy(logits, y):
        """Accuracy (%) from logits of shape (N, C)."""
        preds = logits.argmax(dim=1)
        return (preds == y).float().mean().item() * 100

    def evaluate(self, x, y):
        """
        Compute loss and accuracy in eval mode.

        Returns
        -------
        tuple
            (loss, accuracy_percent)
        """
        self.model.eval()

        with torch.inference_mode():
            logits = self.model(x)
            loss = self.loss_fn(logits, y).item()
            acc = self.accuracy(logits, y)

        return loss, acc

    def train(self):
        """
        Train the model.

        For every epoch:
        1. Forward pass
        2. Compute loss
        3. Backpropagation
        4. Optimizer step
        5. Periodic evaluation (train and test, both in eval mode)

        Returns
        -------
        torch.nn.Module
            Trained model.
        """
        torch.manual_seed(self.random_state)

        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(self.random_state)

        for epoch in range(self.epochs):

            self.model.train()

            logits = self.model(self.x_train)
            loss = self.loss_fn(logits, self.y_train)

            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

            is_display_epoch = (
                epoch % self.display_range == 0
                or epoch == self.epochs - 1
            )

            if is_display_epoch:

                train_loss, train_acc = self.evaluate(
                    self.x_train, self.y_train
                )
                test_loss, test_acc = self.evaluate(
                    self.x_test, self.y_test
                )

                self.history["epoch"].append(epoch)
                self.history["train_loss"].append(train_loss)
                self.history["test_loss"].append(test_loss)
                self.history["train_acc"].append(train_acc)
                self.history["test_acc"].append(test_acc)

                print(
                    f"Epoch [{epoch:4d}/{self.epochs}] | "
                    f"Train Loss: {train_loss:.4f} | "
                    f"Test Loss: {test_loss:.4f} | "
                    f"Train Acc: {train_acc:.2f}% | "
                    f"Test Acc: {test_acc:.2f}%"
                )

        return self.model


if __name__ == "__main__":
    # Quick demo on the Iris dataset (3 classes)
    from sklearn.datasets import load_iris

    X, y = load_iris(return_X_y=True)

    torch.manual_seed(42)  # seed BEFORE creating the model
    model = torch.nn.Sequential(
        torch.nn.Linear(4, 16),
        torch.nn.ReLU(),
        torch.nn.Linear(16, 3),  # raw logits, no Softmax
    )

    trainer = Trainer(
        model=model,
        loss_fn=torch.nn.CrossEntropyLoss(),
        DATA=(X, y),
        epochs=200,
        display_range=20,
        lr=0.01,
    )
    trainer.train()

Epoch [   0/200] | Train Loss: 1.1810 | Test Loss: 1.1825 | Train Acc: 33.33% | Test Acc: 33.33%
Epoch [  20/200] | Train Loss: 0.5949 | Test Loss: 0.5909 | Train Acc: 66.67% | Test Acc: 66.67%
Epoch [  40/200] | Train Loss: 0.3519 | Test Loss: 0.3567 | Train Acc: 96.67% | Test Acc: 96.67%
Epoch [  60/200] | Train Loss: 0.1981 | Test Loss: 0.2064 | Train Acc: 97.50% | Test Acc: 100.00%
Epoch [  80/200] | Train Loss: 0.1216 | Test Loss: 0.1307 | Train Acc: 98.33% | Test Acc: 100.00%
Epoch [ 100/200] | Train Loss: 0.0903 | Test Loss: 0.0972 | Train Acc: 98.33% | Test Acc: 100.00%
Epoch [ 120/200] | Train Loss: 0.0761 | Test Loss: 0.0804 | Train Acc: 97.50% | Test Acc: 100.00%
Epoch [ 140/200] | Train Loss: 0.0684 | Test Loss: 0.0703 | Train Acc: 97.50% | Test Acc: 100.00%
Epoch [ 160/200] | Train Loss: 0.0635 | Test Loss: 0.0637 | Train Acc: 98.33% | Test Acc: 100.00%
Epoch [ 180/200] | Train Loss: 0.0600 | Test Loss: 0.0590 | Train Acc: 98.33% | Test Acc: 100.00%
Epoch [ 199/200] | Trai